[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Murua-Alvaro/urban/blob/feat/denue-colab-workspace/notebooks/denue/00_COLAB_START_HERE.ipynb)

# Urban · DENUE histórico · Colab

Este notebook clona el repositorio, instala el entorno y ejecuta el pipeline reproducible de DENUE.

Mientras el ZIP auditado todavía no esté físicamente en GitHub, el notebook puede pedirte subirlo manualmente **una vez por sesión**. En cuanto el ZIP esté en `data/denue-historico/data/denue/raw/`, la descarga será automática y ya no aparecerá el selector de archivos.


In [ ]:
REPO = 'https://github.com/Murua-Alvaro/urban.git'
BRANCH = 'feat/denue-colab-workspace'
WORKDIR = '/content/urban'

USE_DRIVE_CACHE = False
MANUAL_UPLOAD_IF_GITHUB_MISSING = True
STAGE = 'eda'  # load | validate | clean | eda | spatial | features | model | all
WITH_MORAN = False

print({
    'branch': BRANCH,
    'stage': STAGE,
    'drive_cache': USE_DRIVE_CACHE,
    'manual_upload_fallback': MANUAL_UPLOAD_IF_GITHUB_MISSING,
})


In [ ]:
import os, subprocess, pathlib, shutil

if pathlib.Path(WORKDIR).exists():
    shutil.rmtree(WORKDIR)

subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, WORKDIR],
    check=True,
)
os.chdir(WORKDIR)
print('Repo listo en', os.getcwd())


In [ ]:
import sys, subprocess

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'],
    check=True,
)
print('Dependencias instaladas con', sys.executable)


In [ ]:
from pathlib import Path

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    DENUE_ROOT = Path('/content/drive/MyDrive/urban-denue-cache')
else:
    DENUE_ROOT = Path('/content/urban-denue')

DENUE_ROOT.mkdir(parents=True, exist_ok=True)
print('DENUE_ROOT =', DENUE_ROOT)


## Archivo DENUE

Esta celda primero revisa si el ZIP ya existe y es válido en la caché. Si no existe, comprueba GitHub. Si GitHub todavía no tiene el ZIP, abre el selector de archivos de Colab para que puedas continuar hoy mismo.


In [ ]:
from pathlib import Path
import urllib.error
import urllib.request

from urban_denue.archive import verify_archive
from urban_denue.constants import ARCHIVE_NAME, ARCHIVE_URL

archive_path = DENUE_ROOT / 'raw' / ARCHIVE_NAME
archive_path.parent.mkdir(parents=True, exist_ok=True)

ready = False

if archive_path.exists():
    try:
        verify_archive(archive_path)
        print('ZIP válido encontrado en caché:', archive_path)
        ready = True
    except Exception as exc:
        print('La copia en caché no es válida y se reemplazará:', exc)
        archive_path.unlink(missing_ok=True)

if not ready:
    try:
        req = urllib.request.Request(
            ARCHIVE_URL,
            headers={'User-Agent': 'urban-denue-colab/1.0'},
        )
        with urllib.request.urlopen(req, timeout=30) as response:
            remote_ok = getattr(response, 'status', 200) == 200
        if remote_ok:
            print('ZIP disponible en GitHub. El pipeline lo descargará automáticamente.')
            ready = True
    except urllib.error.HTTPError as exc:
        if exc.code != 404:
            raise
        print('El ZIP todavía NO está en GitHub.')
        print('Ruta esperada:', ARCHIVE_URL)
    except Exception as exc:
        print('No se pudo comprobar GitHub:', repr(exc))

if not ready:
    if not MANUAL_UPLOAD_IF_GITHUB_MISSING:
        raise FileNotFoundError(
            'El ZIP no está en GitHub. Activa MANUAL_UPLOAD_IF_GITHUB_MISSING=True '
            'o súbelo a la rama data/denue-historico.'
        )

    from google.colab import files
    print('Selecciona el ZIP auditado de DENUE en tu computadora.')
    uploaded = files.upload()

    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise ValueError('Sube exactamente un archivo ZIP.')

    uploaded_name = zip_names[0]
    archive_path.write_bytes(uploaded[uploaded_name])

    try:
        verify_archive(archive_path)
    except Exception:
        archive_path.unlink(missing_ok=True)
        raise

    print('ZIP verificado correctamente y guardado como:', archive_path)
    ready = True

assert ready


## Ejecutar pipeline

La salida se transmite línea por línea para que, si hay otro problema, veas el **traceback real** y no solamente `CalledProcessError`.


In [ ]:
import subprocess, sys

cmd = [
    sys.executable,
    'scripts/denue/colab_entrypoint.py',
    '--stage',
    STAGE,
    '--root',
    str(DENUE_ROOT),
]
if WITH_MORAN:
    cmd.append('--with-moran')

print('Ejecutando:', ' '.join(cmd))

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='')

return_code = proc.wait()
if return_code != 0:
    raise RuntimeError(
        f'El pipeline terminó con código {return_code}. '
        'La causa real aparece en las líneas inmediatamente anteriores.'
    )

print('\nPipeline terminado correctamente.')


## Inspección rápida

Las siguientes celdas buscan salidas Parquet/CSV generadas por el pipeline.


In [ ]:
outputs = sorted(DENUE_ROOT.rglob('*.parquet')) + sorted(DENUE_ROOT.rglob('*.csv'))
print(f'{len(outputs)} archivos analíticos encontrados')
for p in outputs[:50]:
    print(p.relative_to(DENUE_ROOT))


In [ ]:
import pandas as pd

candidates = [
    p for p in DENUE_ROOT.rglob('*.parquet')
    if 'municip' in p.name.lower() or 'state' in p.name.lower()
]

if candidates:
    sample = pd.read_parquet(candidates[0])
    display(sample.head(20))
else:
    print('Ejecuta STAGE="eda" o STAGE="all" para generar tablas de EDA.')


## Ejecución completa

Para correr validación, limpieza, EDA, análisis espacial, feature engineering **y modelado**, usa `STAGE = 'all'` y luego **Entorno de ejecución → Ejecutar todas**.

Si usas `USE_DRIVE_CACHE = True`, el ZIP y las salidas pueden persistir entre sesiones de Colab.
